In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import numpy as np

N_PEERS = 300
N_ITERS = 5
T_OFFSET = 10.8
T_END = 300

In [ ]:
def read_data(target: str) -> list[list[tuple[list[float], list[float]]]]:
    result = []
    for seed in range(N_ITERS):
        file_paths = [f'../scale/{target}/{N_PEERS}/{seed}/ifstat_h{i+1}.log' for i in range(N_PEERS)]

        data = []
        for file_path in file_paths:
            time_points = []
            netusages = []
            with open(file_path, 'r') as f:
                # Skip first two title lines
                next(f, None)
                next(f, None)

                for line_no, line in enumerate(f):
                    execution_time = line_no / 10 - T_OFFSET
                    netusage = sum([float(v) for v in line.strip().split()])
                    
                    # skip before burst
                    if execution_time < 0.0:
                        continue
                    
                    time_points.append(execution_time)
                    netusages.append(netusage)
            #data.append((time_points[:(1-10)], np.convolve(netusages, np.ones(10)/10, mode='valid')))
            data.append((time_points, netusages))

        result.append(data)
    return result

In [ ]:
def time_split_sum(time_points: list[float], netusages: list[float]) -> list[float]:
    result = []
    time_now = 0.0
    split_sum = 0.0
    for i in range(len(time_points)):
        t = time_points[i]
        n = netusages[i]

        while t >= time_now + 10:
            time_now += 10
            result.append(split_sum)
            split_sum = 0

        split_sum += n / 8 * 0.1
    return result

In [ ]:
def time_split_netuse_per_peer(data: list[tuple[list[float], list[float]]]) -> list[float]:
    data_hosts = [time_split_sum(t, n) for t, n in data]
    min_length = min([len(h) for h in data_hosts])
    data_hosts_trimmed = [l[:min_length] for l in data_hosts]

    result = []
    for i in range(min_length):
        result.append(sum([v[i] for v in data_hosts_trimmed])/(10 * (i+1)))
    return result

In [ ]:
def parse_all_seeds(target: str) -> tuple[list[int], list[list[float]]]:
    netuse_flow_seeds = [time_split_netuse_per_peer(data) for data in read_data(target)]
    min_length = min([len(n) for n in netuse_flow_seeds])
    netuse_flow_trimmed = [l[:min_length] for l in netuse_flow_seeds]

    result = []
    for i in range(min_length):
        result.append([n[i] for n in netuse_flow_trimmed])
    return [10 * (i+1) for i in range(min_length)], result

In [ ]:
def draw(targets: list[str]):
    fig, ax = plt.subplots(figsize=(9, 5))

    palette = [
        ['#cde4f5', '#185FA5', '#378ADD'],
        ['#f5cdcd', '#a51818', '#dd3737'],
        ['#cdf5d6', '#18a558', '#37dd7a']
    ]

    for idx, target in enumerate(targets):
        size_growth, netuses = parse_all_seeds(target)

        # Draw box plots
        bp = ax.boxplot(netuses, labels=[str(v) for v in size_growth], patch_artist=True,
                        boxprops=dict(facecolor=palette[idx][0], color=palette[idx][1]),
                        medianprops=dict(color=palette[idx][1], linewidth=2),
                        whiskerprops=dict(color=palette[idx][1]),
                        capprops=dict(color=palette[idx][1]),
                        flierprops=dict(markerfacecolor=palette[idx][2], marker='o',
                                    markersize=4, alpha=0.5))

        # # Compute medians and connect with a line
        # medians = [np.median(d) for d in data]
        # x_positions = range(1, len(data) + 1)

        # ax.plot(x_positions, medians,
        #         color='#D85A30', marker='o', linewidth=2,
        #         markersize=6, zorder=5, label='Median trend')

    # Create legend handles
    legend_labels = ['c-s', 'trickle', 'ours']
    legend_handles = [mpatches.Patch(color=palette[i][1], label=legend_labels[i]) 
                      for i in range(len(targets))]

    ax.set_xlabel('Number of Peers')
    ax.set_ylabel('Total Network Usage per Peer')
    ax.legend(handles=legend_handles, loc='upper right')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(f'scale_netuse_cmp.jpg', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
draw(['client-server', 'dev-eval-trickle', 'dev-v2'])